In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [15]:
!pip install torchao -q --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 24.7 MB/s eta 0:00:0000:0100:01


# Q1. Label Encoding
Convert the answer column in train.csv into numeric labels using the following mapping:
A = 0
B = 1
C = 2
D = 3
E = 4

What is the encoded numeric label for the row at index 150?

In [6]:
import pandas as pd
from datasets import load_dataset

ds = load_dataset('csv', data_files='/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv', split='train')

label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
labels = [label_map[a] for a in ds['answer']]
print('Q1 Answer:', labels[150])

Generating train split: 0 examples [00:00, ? examples/s]

Q1 Answer: 2


# Q2. Prompt-Option Formatting
For row index 0, create the Option B input using exactly this format:
str(prompt) + " [SEP] " + str(option_B)

What is the exact character length of this formatted input string?

In [7]:
row0 = ds[0]
formatted = str(row0['prompt']) + ' [SEP] ' + str(row0['B'])
print('Q2 Answer:', len(formatted))

Q2 Answer: 407


# Q3. Single-Row MCQ Tokenization
Using bert-base-uncased, tokenize the five formatted inputs for row index 0 with:
padding = "max_length"
truncation = True
max_length = 128
return_tensors = "pt"

After reshaping for a multiple-choice model, the final input_ids tensor has shape:
[1, 5, 128]

What is the value of the second dimension?

In [8]:
import torch
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
OPTIONS = ['A', 'B', 'C', 'D', 'E']

row0 = ds[0]
inputs_row0 = [str(row0['prompt']) + ' [SEP] ' + str(row0[o]) for o in OPTIONS]

enc = tokenizer(inputs_row0, padding='max_length', truncation=True, max_length=128, return_tensors='pt')
input_ids = enc['input_ids'].unsqueeze(0)  # [1, 5, 128]
print('Q3 Answer (2nd dimension):', input_ids.shape[1])

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Q3 Answer (2nd dimension): 5


# Q4. Batch MCQ Tokenization
Tokenize the first 16 rows of train.csv as multiple-choice examples.
Each row has 5 choices.
Each choice is tokenized to length 128.

The final input_ids tensor has shape:
[16, 5, 128]

How many total token positions are in this tensor?

In [9]:
all_input_ids = []
for i in range(16):
    row = ds[i]
    choices = [str(row['prompt']) + ' [SEP] ' + str(row[o]) for o in OPTIONS]
    enc = tokenizer(choices, padding='max_length', truncation=True, max_length=128, return_tensors='pt')
    all_input_ids.append(enc['input_ids'])

batch_input_ids = torch.stack(all_input_ids)  # [16, 5, 128]
print('Q4 Shape:', tuple(batch_input_ids.shape))
print('Q4 Answer (total token positions):', batch_input_ids.numel())

Q4 Shape: (16, 5, 128)
Q4 Answer (total token positions): 10240


# Q5. Multiple-Choice Logits
Load bert-base-uncased using AutoModelForMultipleChoice.
Tokenize row index 0 as 5 choices and pass it through the model.

The output logits tensor has shape:
[1, 5]

How many logits are produced for one question?

In [10]:
from transformers import AutoModelForMultipleChoice

mc_model = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')
mc_model.eval()

row0 = ds[0]
choices = [str(row0['prompt']) + ' [SEP] ' + str(row0[o]) for o in OPTIONS]
enc = tokenizer(choices, padding='max_length', truncation=True, max_length=128, return_tensors='pt')
input_ids = enc['input_ids'].unsqueeze(0)
attention_mask = enc['attention_mask'].unsqueeze(0)
token_type_ids = enc['token_type_ids'].unsqueeze(0)

with torch.no_grad():
    outputs = mc_model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)

print('Q5 Logits shape:', tuple(outputs.logits.shape))
print('Q5 Answer (num logits):', outputs.logits.shape[1])

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Q5 Logits shape: (1, 5)
Q5 Answer (num logits): 5


# Q6. Supervised Loss Tensor
For row index 0, pass the tokenized 5-choice input into AutoModelForMultipleChoice along with the correct encoded label.

The model returns a scalar loss tensor.

How many dimensions does this loss tensor have?

In [11]:
label_tensor = torch.tensor([labels[0]])

with torch.no_grad():
    outputs_loss = mc_model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        token_type_ids=token_type_ids,
        labels=label_tensor
    )

print('Q6 Loss:', outputs_loss.loss)
print('Q6 Answer (dimensions):', outputs_loss.loss.dim())


Q6 Loss: tensor(1.6243)
Q6 Answer (dimensions): 0


# Q7. LoRA Trainable Parameters
Apply LoRA to the bert-base-uncased multiple-choice model using:
r = 8
lora_alpha = 16
target_modules = ["query", "value"]
lora_dropout = 0.1
bias = "none"
task_type = TaskType.SEQ_CLS

Count trainable parameters using:
sum(p.numel() for p in model.parameters() if p.requires_grad)

How many parameters are trainable?

In [16]:
from peft import get_peft_model, LoraConfig, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

lora_model = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')
lora_model = get_peft_model(lora_model, lora_config)

trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
print('Q7 Answer (trainable params):', trainable)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Skipping import of cpp extensions due to inco

Q7 Answer (trainable params): 295681


# Q8. Hugging Face Dataset Preparation
Create a Hugging Face Dataset from the first 100 rows of train.csv.

For each row, create:
input_ids with shape [5, 128]
attention_mask with shape [5, 128]
labels as the encoded answer label

For the first dataset item, input_ids has shape:
[5, 128]

How many tokenized choices are stored in input_ids?

In [17]:
from datasets import Dataset

def preprocess(idx):
    row = ds[idx]
    choices = [str(row['prompt']) + ' [SEP] ' + str(row[o]) for o in OPTIONS]
    enc = tokenizer(choices, padding='max_length', truncation=True, max_length=128)
    return {
        'input_ids': enc['input_ids'],           # [5, 128]
        'attention_mask': enc['attention_mask'],  # [5, 128]
        'labels': labels[idx]
    }

data = [preprocess(i) for i in range(100)]
hf_ds = Dataset.from_list(data)
hf_ds.set_format(type='torch')

print('Q8 input_ids shape:', hf_ds[0]['input_ids'].shape)
print('Q8 Answer (tokenized choices):', hf_ds[0]['input_ids'].shape[0])

Q8 input_ids shape: torch.Size([5, 128])
Q8 Answer (tokenized choices): 5


# Q9. Tiny LoRA Fine-Tuning
Fine-tune a LoRA multiple-choice model on the first 32 rows using Hugging Face Trainer.

Use the following settings:
max_length = 64
per_device_train_batch_size = 4
gradient_accumulation_steps = 1
max_steps = 4

What is the final global_step reported by the Trainer?

In [18]:
from transformers import TrainingArguments, Trainer

train_data = [preprocess(i) for i in range(32)]
train_ds_32 = Dataset.from_list(train_data)
train_ds_32.set_format(type='torch')

training_args = TrainingArguments(
    output_dir='./lora_mc_output',
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    report_to='none',
    save_steps=999,
)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=train_ds_32,
)

train_result = trainer.train()
print('Q9 Answer (global_step):', train_result.global_step)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
1,3.128556
2,3.164257
3,3.202046
4,3.151618


Q9 Answer (global_step): 4


# Q10. Probability Assigned to Option E After Fine-Tuning
Using the fine-tuned LoRA model from Q9, run inference on row index 0 and apply softmax to the logits.

What is the probability assigned to Option E?

Round your answer to 4 decimal places.

In [20]:
import torch.nn.functional as F

device = next(lora_model.parameters()).device
lora_model.eval()

row0 = ds[0]
choices = [str(row0['prompt']) + ' [SEP] ' + str(row0[o]) for o in OPTIONS]
enc = tokenizer(choices, padding='max_length', truncation=True, max_length=128, return_tensors='pt')

input_ids_inf      = enc['input_ids'].unsqueeze(0).to(device)
attention_mask_inf = enc['attention_mask'].unsqueeze(0).to(device)
token_type_ids_inf = enc['token_type_ids'].unsqueeze(0).to(device)

with torch.no_grad():
    out = lora_model(input_ids=input_ids_inf, attention_mask=attention_mask_inf, token_type_ids=token_type_ids_inf)

probs = F.softmax(out.logits, dim=-1)[0]
print('All probs (A-E):', [round(p.item(), 4) for p in probs])
print('Q10 Answer (Option E prob):', round(probs[4].item(), 4))

All probs (A-E): [0.1986, 0.2037, 0.1999, 0.1992, 0.1986]
Q10 Answer (Option E prob): 0.1986
